# **NLP Practical 5 — Vector Semantics & Word Embeddings (Word2Vec)**

---

**Name: Shubham Sarwar**  
**PRN: 202301040127**

**Aim:** Represent words as **dense numeric vectors** using Word2Vec, find semantically
similar words, perform vector arithmetic (king − man + woman ≈ queen), and visualize
the word vectors in 2-D using t-SNE.

**Core idea — Distributional Semantics:**  
*"You shall know a word by the company it keeps"* (J.R. Firth, 1957).  
Words used in similar contexts (e.g. *king/queen*, *cat/dog*) should end up with
similar vectors. Word2Vec is the most famous algorithm that learns these vectors.


## **1. Install Required Libraries**

In [ ]:
# gensim  -> contains the Word2Vec implementation
# scikit-learn -> contains the t-SNE algorithm for 2-D visualization
# matplotlib -> for plotting
!pip install gensim matplotlib scikit-learn

In [ ]:
from gensim.models import Word2Vec        # the model itself
from sklearn.manifold import TSNE             # t-SNE = dimensionality-reduction technique
import matplotlib.pyplot as plt
import numpy as np

## **2. Prepare the Training Corpus**

Word2Vec expects a **list of tokenized sentences**.  
The MORE & BIGGER the sentences the better the model — Word2Vec learns from
words appearing TOGETHER in context. Tiny sentences of 3 words give weak results.


In [ ]:
# A richer corpus where related words occur together repeatedly
sentences = [
    ["the","king","ruled","the","kingdom","with","wisdom","and","power"],
    ["the","queen","ruled","beside","the","king","in","the","royal","palace"],
    ["the","king","and","queen","lived","in","a","grand","palace"],
    ["a","man","walked","into","the","kingdom","to","meet","the","king"],
    ["a","woman","entered","the","palace","to","speak","with","the","queen"],
    ["the","prince","and","princess","played","in","the","royal","garden"],

    ["paris","is","the","capital","city","of","france","in","europe"],
    ["london","is","the","capital","city","of","england","in","europe"],
    ["berlin","is","the","capital","city","of","germany","in","europe"],
    ["rome","is","the","capital","city","of","italy","in","europe"],
    ["france","and","germany","are","european","countries","with","rich","history"],
    ["england","and","italy","are","european","countries","with","ancient","cities"],

    ["the","dog","and","the","cat","are","common","pet","animals","at","home"],
    ["dogs","love","to","play","with","balls","in","the","park"],
    ["cats","love","to","sleep","on","warm","sofas","at","home"],

    ["apple","and","orange","are","sweet","juicy","fruits","grown","on","trees"],
    ["banana","and","mango","are","tropical","fruits","loved","by","children"],

    ["python","and","java","are","popular","programming","languages","used","by","developers"],
    ["c","and","cpp","are","also","programming","languages","used","in","systems"],

    ["the","car","and","the","bus","are","types","of","road","vehicles"],
    ["a","bike","is","a","fast","two","wheeled","vehicle","for","short","trips"],

    ["the","teacher","taught","students","in","the","school","classroom","every","morning"],
    ["the","student","studied","hard","at","school","to","pass","the","exam"],

    ["the","sun","rises","in","the","east","and","sets","in","the","west"],
    ["the","moon","shines","at","night","in","the","dark","sky"],
    ["stars","twinkle","brightly","in","the","clear","night","sky"],
]

print(f"Number of sentences: {len(sentences)}")
print(f"Total tokens       : {sum(len(s) for s in sentences)}")

## **3. Train the Word2Vec Model**

Important parameters:
- `vector_size` → dimension of each word vector (commonly 50–300). Larger = richer but needs more data.
- `window` → how many words on each side count as "context".
- `min_count` → ignore words that appear fewer than this many times.
- `sg` → algorithm:
  - `sg=0` → **CBOW** (Continuous Bag-of-Words): predicts the *current* word from its context. Faster, works better with small data.
  - `sg=1` → **Skip-gram**: predicts the *context* from the current word. Slower, better for rare words.
- `epochs` → how many times we pass through the data. More epochs ⇒ better-converged vectors.
- `seed` → makes the result reproducible.


In [ ]:
model = Word2Vec(
    sentences=sentences,
    vector_size=50,    # each word becomes a 50-number vector
    window=5,          # look at 5 words to the left/right as context
    min_count=1,       # include every word (even those appearing only once)
    sg=0,              # 0 = CBOW (better for our small corpus)
    epochs=200,        # train 200 times over the data so vectors converge
    seed=42            # reproducibility
)

print(f"Vocabulary size: {len(model.wv)} unique words")
print(f"Vector size    : {model.vector_size}")

## **4. Inspect a Word Vector**

In [ ]:
# Every word now has a numeric vector. Let's look at one:
print("Vector for 'king' (50 numbers):")
print(model.wv['king'])

print("\nShape of the vector:", model.wv['king'].shape)

## **5. Find Most Similar Words**

`model.wv.most_similar(word)` returns the words whose vectors are closest in direction
to the given word (using **cosine similarity** — values close to 1.0 mean very similar).


In [ ]:
words_to_check = ["king", "france", "dog", "python", "school"]

for word in words_to_check:
    print(f"\nWords most similar to '{word}':")
    for similar_word, score in model.wv.most_similar(word, topn=5):
        print(f"   {similar_word:15s} (similarity = {score:.3f})")

## **6. Vector Arithmetic — The Famous "king − man + woman ≈ queen"**

Word2Vec captures relationships *geometrically*. Subtracting and adding vectors
lets us solve analogies:

`positive=['king','woman'], negative=['man']`  
means: take vector(king) + vector(woman) − vector(man), then find the word closest
to that result.


In [ ]:
print("king - man + woman = ?")
result = model.wv.most_similar(positive=['king', 'woman'], negative=['man'], topn=5)
for w, s in result:
    print(f"   {w}: {s:.3f}")

print("\nparis - france + england = ?  (expecting 'london')")
result = model.wv.most_similar(positive=['paris', 'england'], negative=['france'], topn=5)
for w, s in result:
    print(f"   {w}: {s:.3f}")

## **7. Direct Similarity Between Two Words**

`model.wv.similarity(w1, w2)` returns just one number — the cosine similarity.


In [ ]:
pairs = [
    ("king", "queen"),
    ("dog", "cat"),
    ("paris", "london"),
    ("apple", "orange"),
    ("king", "apple"),    # unrelated -> should be lower
]

for a, b in pairs:
    sim = model.wv.similarity(a, b)
    print(f"similarity({a:8s}, {b:8s}) = {sim:.3f}")

## **8. Visualize Word Vectors with t-SNE**

Each vector has 50 dimensions — too many to plot. **t-SNE** (t-Distributed Stochastic
Neighbour Embedding) squeezes them down to 2 dimensions while keeping similar words
close together. Then we can plot them on a normal X-Y graph.


In [ ]:
# Collect all words & their vectors
words   = list(model.wv.index_to_key)
vectors = np.array([model.wv[w] for w in words])

# perplexity must be < number of words; pick a small safe value
perp = min(15, max(2, len(words) - 1))

# Reduce 50-D -> 2-D using t-SNE
tsne = TSNE(n_components=2, perplexity=perp, random_state=42, init="random", learning_rate="auto")
vectors_2d = tsne.fit_transform(vectors)

# Plot
plt.figure(figsize=(12, 8))
plt.scatter(vectors_2d[:, 0], vectors_2d[:, 1], s=20, c="steelblue")

# Annotate every point with the actual word
for i, w in enumerate(words):
    plt.annotate(w, (vectors_2d[i, 0], vectors_2d[i, 1]),
                 fontsize=9, alpha=0.85)

plt.title("Word2Vec Embeddings Visualized with t-SNE")
plt.xlabel("Dim 1")
plt.ylabel("Dim 2")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

### **Observation**

Notice in the t-SNE plot how words from the same theme cluster together:
- royalty terms (*king, queen, palace, royal*) form one group,
- countries/cities (*paris, london, france, england*) form another,
- animals (*dog, cat, pet*) cluster, fruits cluster, programming languages cluster, etc.

This shows that Word2Vec successfully learned **semantic similarity** purely from
the patterns of word co-occurrence in the corpus — no manual labelling needed.
